# Final Project 2 error analysis

Run Final_Project_Model.ipynb first with application_train.csv in the same directory. This notebook uses its generated local .project2_runtime predictions, checks them against the submitted CSV, and preserves your comments. Only Final_Error_Analysis.csv is a submission spreadsheet; supporting diagnostics stay in the hidden local runtime folder.


In [1]:
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score,confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
def fairness_tables(predictions, attributes, cutoff):
    """Audit TARGET=1 difficulty; undefined rates stay NaN, never zero."""
    data = predictions.merge(attributes[['SK_ID_CURR', 'CODE_GENDER', 'DAYS_BIRTH']],
                             on='SK_ID_CURR', validate='one_to_one')
    data['age_group'] = pd.cut(-data.DAYS_BIRTH / 365.25, [0, 30, 45, 60, np.inf],
        right=False, labels=['Under 30', '30 to under 45', '45 to under 60', '60 and older'])
    data['label'] = data.predicted_probability.ge(cutoff)
    rows = []
    for attribute in ['CODE_GENDER', 'age_group']:
        for group, part in data.groupby(attribute, observed=True, dropna=False):
            actual = part.actual_TARGET.eq(1); label = part.label
            tp=int((actual & label).sum()); fn=int((actual & ~label).sum())
            fp=int((~actual & label).sum()); tn=int((~actual & ~label).sum())
            counts = {'false_positive_rate': (fp, fp+tn), 'recall': (tp, tp+fn),
                      'precision': (tp, tp+fp), 'npv': (tn, tn+fn),
                      'specificity': (tn, tn+fp), 'false_negative_rate': (fn, fn+tp),
                      'favorable_prediction_rate': (tn+fn, len(part)),
                      'difficulty_prevalence': (tp+fn, len(part))}
            row=dict(attribute=attribute, group=str(group), n=len(part),
                     positives=tp+fn, negatives=tn+fp, tp=tp, fn=fn, fp=fp, tn=tn)
            for metric, (success, total) in counts.items():
                rate=success/total if total else np.nan
                row[metric]=rate; row[metric+'_denominator']=total
                # Descriptive 95% Wilson interval; does not account for repeated dev selection.
                if total:
                    z=1.959963984540054; den=1+z*z/total
                    center=(rate+z*z/(2*total))/den
                    half=z*np.sqrt(rate*(1-rate)/total+z*z/(4*total*total))/den
                    row[metric+'_lower']=max(0.,center-half); row[metric+'_upper']=min(1.,center+half)
                else:
                    row[metric+'_lower']=np.nan; row[metric+'_upper']=np.nan
            row['small_group_warning']=min(tp+fn,tn+fp,tp+fp,tn+fn)<30
            rows.append(row)
    groups=pd.DataFrame(rows)
    gaps=[]
    for attribute in groups.attribute.unique():
        part=groups[groups.attribute.eq(attribute)]
        for metric in ['false_positive_rate','recall','precision','npv','favorable_prediction_rate','specificity']:
            eligible=part[part[metric+'_denominator'].ge(30)&part[metric].notna()]
            if len(eligible)>=2:
                lo=eligible.loc[eligible[metric].idxmin()]; hi=eligible.loc[eligible[metric].idxmax()]
                gaps.append(dict(attribute=attribute,metric=metric,eligible_groups=len(eligible),
                    excluded_groups='; '.join(part.loc[~part.index.isin(eligible.index),'group']),
                    lowest_group=lo['group'],highest_group=hi['group'],
                    minimum=lo[metric],maximum=hi[metric],gap_pp=100*(hi[metric]-lo[metric])))
    return groups, pd.DataFrame(gaps)
DATA_DIR=Path.cwd()
input_dir=DATA_DIR/'.project2_runtime/selected'
output_dir=DATA_DIR/'.project2_runtime/error_analysis'
output_dir.mkdir(parents=True,exist_ok=True)
assert (input_dir/'model_info.json').exists(), 'Run Final_Project_Model.ipynb first in this directory.'
model_info=json.loads((input_dir/'model_info.json').read_text())
run_name=model_info['model_id'];threshold=model_info['threshold']
assert run_name=='final_application_peer_seed42' and threshold==.167
manifest=json.loads((input_dir/'dev_split_manifest.json').read_text());eyeball_ids=set(manifest['eyeball_ids'])
blackbox=pd.read_json(input_dir/'blackbox_predictions.json')
eyeball_predictions=pd.read_json(input_dir/'eyeball_predictions.json')
assert len(blackbox)==61002 and len(eyeball_predictions)==500
assert set(eyeball_predictions.SK_ID_CURR)==eyeball_ids
assert set(blackbox.SK_ID_CURR).isdisjoint(eyeball_ids)
raw=pd.read_csv('application_train.csv')
review=eyeball_predictions.merge(raw[raw.SK_ID_CURR.isin(eyeball_ids)],on='SK_ID_CURR',validate='one_to_one')
assert review.actual_TARGET.eq(review.TARGET).all()
assert np.array_equal(review.predicted_label,(review.predicted_probability>=threshold).astype(int))
review['age_years']=-review.DAYS_BIRTH/365.25
review['distance_from_threshold']=review.predicted_probability-threshold
review['outcome']=np.select([review.actual_TARGET.eq(1)&review.predicted_label.eq(0),review.actual_TARGET.eq(0)&review.predicted_label.eq(1),review.actual_TARGET.eq(1)&review.predicted_label.eq(1)],['False negative','False positive','True positive'],default='True negative')

submitted=pd.read_csv('Final_Error_Analysis.csv')
current_errors=review[review.actual_TARGET.ne(review.predicted_label)].set_index('SK_ID_CURR')
expected=submitted.set_index('SK_ID_CURR')
assert set(current_errors.index)==set(expected.index), 'Rerun errors differ from the submitted model; preserve CSV and investigate.'
assert np.allclose(current_errors.loc[expected.index,'predicted_probability'],expected.predicted_probability,atol=1e-7), 'Predictions differ from the final review.'


<ipython-input-1-7421adf48a68>:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  review['age_years']=-review.DAYS_BIRTH/365.25
<ipython-input-1-7421adf48a68>:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  review['distance_from_threshold']=review.predicted_probability-threshold
<ipython-input-1-7421adf48a68>:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. T

## 1 Error inventory

In [2]:
def metrics(frame):
    y=frame.actual_TARGET;p=frame.predicted_probability;label=p>=threshold
    tn,fp,fn,tp=confusion_matrix(y,label,labels=[0,1]).ravel()
    return dict(n=len(frame),auc=roc_auc_score(y,p),accuracy=accuracy_score(y,label),
        precision=precision_score(y,label,zero_division=0),recall=recall_score(y,label,zero_division=0),
        f1=f1_score(y,label,zero_division=0),false_positive_rate=fp/(fp+tn),
        true_negatives=int(tn),false_positives=int(fp),false_negatives=int(fn),true_positives=int(tp))
blackbox_metrics=metrics(blackbox)
(output_dir/'blackbox_metrics.json').write_text(json.dumps(blackbox_metrics,indent=2))
display(pd.DataFrame([blackbox_metrics],index=['Blackbox dev']).round(4))
print('Eyeball review workload:')
display(review.outcome.value_counts().rename_axis('outcome').to_frame('records'))
errors=review[review.predicted_label.ne(review.actual_TARGET)].sort_values(['outcome','SK_ID_CURR']).reset_index(drop=True)
assert set(errors.SK_ID_CURR).issubset(eyeball_ids)
errors.to_csv(output_dir/'all_eyeball_errors.csv',index=False)

review.to_csv(output_dir/'eyeball_predictions.csv',index=False)


Eyeball review workload:


,n,auc,accuracy,precision,recall,f1,false_positive_rate,true_negatives,false_positives,false_negatives,true_positives
Blackbox dev,61002,0.7547,0.8525,0.2447,0.3961,0.3026,0.1074,50056,6021,2974,1951


,records
outcome,
True negative,407
False positive,53
False negative,28
True positive,12


## 2 Case notes and correct-case comparisons

In [3]:
def tags_for(row):
    tags=[]
    if abs(row.distance_from_threshold)<=.02: tags.append('Near threshold')
    if row.age_years>=45 and row.EXT_SOURCE_2>=.5 and row.outcome=='False negative': tags.append('Older high-score missed difficulty')
    if row.age_years<30 and row.EXT_SOURCE_3<.25 and row.outcome=='False positive': tags.append('Younger low-score false alarm')
    if row[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].isna().any(): tags.append('Missing external score')
    if row.EXT_SOURCE_2<.11: tags.append('Very low EXT_SOURCE_2')
    if row.AMT_INCOME_TOTAL>0 and row.AMT_CREDIT/row.AMT_INCOME_TOTAL>4: tags.append('Credit exceeds four times income')
    if (row.EXT_SOURCE_2>=.5 and row.EXT_SOURCE_3<.25) or (row.EXT_SOURCE_3>=.5 and row.EXT_SOURCE_2<.25): tags.append('Conflicting external scores')
    if row.DAYS_EMPLOYED==365243: tags.append('Employment sentinel in raw data')
    return '; '.join(tags) if tags else 'No predefined pattern'

def fmt(value): return 'missing' if pd.isna(value) else f'{value:.3f}'
rows=[];comparisons=[]
for _,row in errors.iterrows():
    missing=[c for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'] if pd.isna(row[c])]
    controls=review[review.actual_TARGET.eq(row.actual_TARGET)&review.predicted_label.eq(review.actual_TARGET)].copy()
    controls['gap']=(controls.EXT_SOURCE_2-row.EXT_SOURCE_2).abs()
    controls=controls[controls.gap.le(.05)].sort_values(['gap','SK_ID_CURR'])
    match=controls.iloc[0] if len(controls) else None
    comparison='No correct case with the same actual outcome within 0.05 on EXT_SOURCE_2 in this eyeball pool.'
    if match is not None:
        comparison=(f'Correct case {int(match.SK_ID_CURR)}: EXT_SOURCE_2 {fmt(match.EXT_SOURCE_2)} versus {fmt(row.EXT_SOURCE_2)}; '
                    f'EXT_SOURCE_3 {fmt(match.EXT_SOURCE_3)} versus {fmt(row.EXT_SOURCE_3)}; '
                    f'age {match.age_years:.1f} versus {row.age_years:.1f}. Other inputs differ; this is not a causal comparison.')
        comparisons.append(dict(error_id=int(row.SK_ID_CURR),control_id=int(match.SK_ID_CURR),
            actual_TARGET=int(row.actual_TARGET),score_gap=float(match.gap),
            error_probability=float(row.predicted_probability),control_probability=float(match.predicted_probability)))
    annuity_ratio=row.AMT_ANNUITY/row.AMT_INCOME_TOTAL if row.AMT_INCOME_TOTAL>0 else np.nan
    credit_ratio=row.AMT_CREDIT/row.AMT_INCOME_TOTAL if row.AMT_INCOME_TOTAL>0 else np.nan
    tags=tags_for(row)
    suggestions=[]
    if 'Near threshold' in tags: suggestions.append('Sensitive to cutoff; evaluate cost tradeoff using blackbox only.')
    if 'Older high-score' in tags: suggestions.append('Missed difficulty despite relatively high EXT_SOURCE_2; score does not determine the outcome.')
    if 'Younger low-score' in tags: suggestions.append('False alarm despite low EXT_SOURCE_3; compare this subgroup with the prior model on blackbox.')
    if missing: suggestions.append('Missing external information may be relevant; compare with correctly classified eyeball cases before attributing the error to missingness.')
    if not suggestions: suggestions.append('No clear explanation from the predefined properties; a label-prediction disagreement alone does not establish a cause.')
    record={c:row[c] for c in ['SK_ID_CURR','actual_TARGET','predicted_label','predicted_probability','outcome','age_years','CODE_GENDER',
        'AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','DAYS_EMPLOYED','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']}
    record.update(model_run=run_name,partition='eyeball',threshold=threshold,distance_from_threshold=row.distance_from_threshold,
        annuity_to_income=annuity_ratio,credit_to_income=credit_ratio,employment_placeholder=bool(row.DAYS_EMPLOYED==365243),
        missing_external_scores='; '.join(missing),error_categories=tags,
        observation=(f'{row.outcome}: actual TARGET {int(row.actual_TARGET)}, score {row.predicted_probability:.1%}, cutoff {threshold:.1%}; '
                     f'age {row.age_years:.1f}. EXT_SOURCE_1={fmt(row.EXT_SOURCE_1)}, EXT_SOURCE_2={fmt(row.EXT_SOURCE_2)}, EXT_SOURCE_3={fmt(row.EXT_SOURCE_3)}.'),
        suspected_pattern=' '.join(suggestions),comparison_applicant_id=int(match.SK_ID_CURR) if match is not None else '',
        comparison_observation=comparison,proposed_check='Check draft against raw features and comparison; count recurring patterns in eyeball, evaluate predeclared changes on blackbox.',
        proposed_change='No individual fix established; use the controlled intervention comparison.',
        result_after_change='Model-level comparison is in Final_Project_Model.ipynb; this row remains an error of the selected final model.',
        dataset_issue_evidence='No demonstrated label corruption; missing fields or an error alone do not justify relabeling.',
        reviewer='',team_review_status='Pending team review',team_comment='',draft_origin='Assistant-generated factual draft; hypotheses unconfirmed')
    rows.append(record)
worksheet=pd.DataFrame(rows)
review_path=DATA_DIR/'Final_Error_Analysis.csv'
# Export only the review columns; detailed drafts stay in the local diagnostics.
worksheet.to_csv(output_dir/'detailed_error_drafts.csv',index=False)
columns=['SK_ID_CURR','actual_TARGET','predicted_label','predicted_probability','outcome','age_years','CODE_GENDER','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','annuity_to_income','credit_to_income','employment_placeholder','error_categories','comments']
worksheet['comments']=''
if review_path.exists():
    existing=pd.read_csv(review_path,keep_default_na=False).set_index('SK_ID_CURR')
    assert set(existing.index)==set(worksheet.SK_ID_CURR), 'Model errors changed; preserve the review.'
    comment_column='comments' if 'comments' in existing else 'team_comment'
    if comment_column in existing:
        worksheet['comments']=worksheet.SK_ID_CURR.map(existing[comment_column]).fillna('')
worksheet=worksheet[columns]
worksheet.to_csv(review_path,index=False)
pd.DataFrame(comparisons,columns=['error_id','control_id','actual_TARGET','score_gap','error_probability','control_probability']).to_csv(output_dir/'correct_case_comparisons.csv',index=False)
assert len(worksheet)==len(errors) and worksheet.SK_ID_CURR.is_unique
assert len(worksheet.columns)==15
print('Drafted all',len(worksheet),'eyeball errors. Human verification remains pending.')
display(worksheet[['SK_ID_CURR','outcome','error_categories','comments']])


Drafted all 81 eyeball errors. Human verification remains pending.


,SK_ID_CURR,outcome,error_categories,comments
0,131596,False negative,Older high-score missed difficulty; Credit exc...,
1,160093,False negative,Older high-score missed difficulty; Missing ex...,
2,162699,False negative,Older high-score missed difficulty; Missing ex...,
3,163613,False negative,Missing external score,
4,174273,False negative,Older high-score missed difficulty,
...,...,...,...,...
76,407637,False positive,No predefined pattern,
77,407690,False positive,No predefined pattern,
78,412964,False positive,Near threshold; Missing external score,
79,421954,False positive,Near threshold; Missing external score,


## 3 Patterns with denominators

In [4]:
pattern_rows=[]
for name,mask in [
    ('Older high-score actual difficulty',review.age_years.ge(45)&review.EXT_SOURCE_2.ge(.5)&review.actual_TARGET.eq(1)),
    ('Younger low-score actual no difficulty',review.age_years.lt(30)&review.EXT_SOURCE_3.lt(.25)&review.actual_TARGET.eq(0)),
    ('Any external score missing',review[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].isna().any(axis=1)),
    ('All external scores present',review[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].notna().all(axis=1)),
    ('Within 0.02 of threshold',review.distance_from_threshold.abs().le(.02))]:
    group=review[mask];count=int(group.actual_TARGET.ne(group.predicted_label).sum())
    pattern_rows.append(dict(pattern=name,eligible_records=len(group),errors=count,error_rate=count/len(group) if len(group) else np.nan))
patterns=pd.DataFrame(pattern_rows)
patterns.to_csv(output_dir/'eyeball_pattern_summary.csv',index=False)
display(patterns.round(4))
print('Priorities: investigate repeated false alarms and missed difficulties; business costs have not been specified.')


Priorities: investigate repeated false alarms and missed difficulties; business costs have not been specified.


,pattern,eligible_records,errors,error_rate
0,Older high-score actual difficulty,12,11,0.9167
1,Younger low-score actual no difficulty,13,5,0.3846
2,Any external score missing,324,48,0.1481
3,All external scores present,176,33,0.1875
4,Within 0.02 of threshold,32,8,0.2500


## 4 Fairness and provenance

In [5]:
fairness_summary, fairness_gaps = fairness_tables(blackbox, raw, threshold)
fairness_summary.to_csv(output_dir/'fairness_metrics.csv',index=False)
fairness_gaps.to_csv(output_dir/'fairness_disparities.csv',index=False)
display(fairness_summary[['attribute','group','n','false_positive_rate','recall','precision','npv','favorable_prediction_rate','small_group_warning']].round(4))
display(fairness_gaps.round(3))
provenance=dict(model_info=model_info,eyeball_count=len(review),eyeball_error_count=len(errors),blackbox_count=len(blackbox),
    manual_review_complete=False,selection_partition='blackbox',case_inspection_partition='eyeball',
    prediction_sha256=hashlib.sha256((input_dir/'eyeball_predictions.json').read_bytes()).hexdigest(),
    historical_limitation='Earlier models and thresholds used full dev. This revision does not erase exposure.')
(output_dir/'workflow_provenance.json').write_text(json.dumps(provenance,indent=2))
(DATA_DIR/'.project2_runtime/error_analysis_outputs.json').write_text(json.dumps({'output_dir':str(output_dir.relative_to(DATA_DIR)),'review_csv':str(review_path.relative_to(DATA_DIR))},indent=2))
print('Review CSV:',review_path)


Review CSV: /private/tmp/project2_four_file_check/Final_Error_Analysis.csv


,attribute,group,n,false_positive_rate,recall,precision,npv,favorable_prediction_rate,small_group_warning
0,CODE_GENDER,F,40195,0.0890,0.3567,0.2323,0.9494,0.8923,False
1,CODE_GENDER,M,20806,0.1441,0.4489,0.2596,0.9324,0.8251,False
2,CODE_GENDER,XNA,1,1.0000,NaN,0.0000,NaN,0.0000,True
3,age_group,Under 30,8900,0.2202,0.5511,0.2420,0.9316,0.7424,False
4,age_group,30 to under 45,24675,0.1235,0.4230,0.2554,0.9382,0.8492,False
5,age_group,45 to under 60,20531,0.0706,0.3037,0.2311,0.9503,0.9142,False
6,age_group,60 and older,6896,0.0242,0.1175,0.1970,0.9563,0.9713,False


,attribute,metric,eligible_groups,excluded_groups,lowest_group,highest_group,minimum,maximum,gap_pp
0,CODE_GENDER,false_positive_rate,2,XNA,F,M,0.089,0.144,5.515
1,CODE_GENDER,recall,2,XNA,F,M,0.357,0.449,9.219
2,CODE_GENDER,precision,2,XNA,F,M,0.232,0.260,2.734
3,CODE_GENDER,npv,2,XNA,M,F,0.932,0.949,1.700
4,CODE_GENDER,favorable_prediction_rate,2,XNA,M,F,0.825,0.892,6.720
5,CODE_GENDER,specificity,2,XNA,M,F,0.856,0.911,5.515
6,age_group,false_positive_rate,4,,60 and older,Under 30,0.024,0.220,19.597
7,age_group,recall,4,,60 and older,Under 30,0.117,0.551,43.367
8,age_group,precision,4,,60 and older,30 to under 45,0.197,0.255,5.841
9,age_group,npv,4,,Under 30,60 and older,0.932,0.956,2.467


## 5 Changes from the previous model

In [6]:
# Recorded comparison from the completed baseline experiment; no old model files are needed.
changes={'previous_errors': 95, 'final_errors': 81, 'corrected': 21, 'new_errors': 7, 'persistent_errors': 74, 'correct_both': 398}
print(changes)
print('Historical comparison: 21 corrected errors, 7 new errors, 74 persistent. This did not select the model.')


{'previous_errors': 95, 'final_errors': 81, 'corrected': 21, 'new_errors': 7, 'persistent_errors': 74, 'correct_both': 398}
Historical comparison: 21 corrected errors, 7 new errors, 74 persistent. This did not select the model.


## 6 Recheck the recorded hypotheses on blackbox aggregates

These hypotheses were developed for the previous model. Their cutoffs and definitions remain fixed here; results describe whether patterns persist after replacement. This is an exploratory recheck on previously used blackbox data, not independent confirmation. Test predictions are not loaded. H1 low EXT2 overprediction; H2 credit/income above4 conditional on score bands; H3 young-group overprediction; H4 older high-EXT2 misses; H5 younger low-EXT3 false alarms; H6 missing scores; H7 threshold proximity; H8 conflicting scores; H9 raw employment sentinel.


In [7]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd()
OUT=ROOT/'.project2_runtime/hypotheses'
OUT.mkdir(parents=True,exist_ok=True)
MODEL=ROOT/'.project2_runtime/selected'
info=json.loads((MODEL/'model_info.json').read_text())
pred=pd.read_json(MODEL/'blackbox_predictions.json')
eye_ids=set(json.loads((MODEL/'dev_split_manifest.json').read_text())['eyeball_ids'])
assert len(pred)==61002 and set(pred.SK_ID_CURR).isdisjoint(eye_ids)
columns=['SK_ID_CURR','TARGET','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3','AMT_CREDIT','AMT_INCOME_TOTAL','DAYS_BIRTH']
raw=pd.read_csv(ROOT/'application_train.csv',usecols=columns)
d=pred.merge(raw,on='SK_ID_CURR',validate='one_to_one')
assert d.actual_TARGET.eq(d.TARGET).all() and d.predicted_probability.between(0,1).all()
d['residual']=d.predicted_probability-d.TARGET
d['flagged']=d.predicted_probability.ge(info['threshold'])
d['age']=-d.DAYS_BIRTH/365.25
z=1.959963984540054

def calibration(g):
    n=len(g);r=g.residual
    se=r.std(ddof=1)/np.sqrt(n) if n>1 else np.nan
    neg=g.TARGET.eq(0);pos=g.TARGET.eq(1)
    return dict(n=n,events=int(pos.sum()),mean_prediction=g.predicted_probability.mean(),
        observed_rate=g.TARGET.mean(),calibration_gap=r.mean(),gap_lower=r.mean()-z*se,gap_upper=r.mean()+z*se,
        false_positive_rate=g.loc[neg,'flagged'].mean(),recall=g.loc[pos,'flagged'].mean(),
        negatives=int(neg.sum()),false_positives=int(g.loc[neg,'flagged'].sum()),small_group=n<100)

def save_table(name,rows):
    frame=pd.DataFrame(rows)
    frame.to_json(OUT/(name+'.json'),orient='records',indent=2)
    print('\n'+name)
    display(frame.round(5))
    return frame

h1=save_table('H1_external_score_calibration',[
    dict(group=name,**calibration(d[mask])) for name,mask in [
        ('EXT2 < 0.11',d.EXT_SOURCE_2.lt(.11)),
        ('0.11 <= EXT2 < 0.25',d.EXT_SOURCE_2.ge(.11)&d.EXT_SOURCE_2.lt(.25)),
        ('0.25 <= EXT2 < 0.50',d.EXT_SOURCE_2.ge(.25)&d.EXT_SOURCE_2.lt(.5)),
        ('EXT2 >= 0.50',d.EXT_SOURCE_2.ge(.5)),
        ('EXT2 missing',d.EXT_SOURCE_2.isna())]])

valid=d.AMT_INCOME_TOTAL.gt(0)&np.isfinite(d.AMT_INCOME_TOTAL)&np.isfinite(d.AMT_CREDIT)
ratio=d.loc[valid].copy()
ratio['credit_income_ratio']=ratio.AMT_CREDIT/ratio.AMT_INCOME_TOTAL
ratio['high_ratio']=ratio.credit_income_ratio.gt(4)
h2_context=save_table('H2_ratio_context',[dict(group=name,**calibration(g)) for name,g in [
    ('Credit/income <= 4',ratio[~ratio.high_ratio]),('Credit/income > 4',ratio[ratio.high_ratio])]])

def adjusted_comparison(width):
    neg=ratio[ratio.TARGET.eq(0)].copy()
    bands=[]
    for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
        col=c+'_band';bands.append(col)
        assert neg[c].dropna().between(0,1).all()
        neg[col]=np.floor(neg[c].clip(upper=1-1e-12)/width).fillna(-1).astype(int)
    grouped=neg.groupby(bands+['high_ratio']).flagged.agg(['size','sum']).unstack('high_ratio')
    # Reindex makes absent ratio groups explicit before restricting to overlap.
    grouped=grouped.reindex(columns=pd.MultiIndex.from_product([['size','sum'],[False,True]]))
    keep=(grouped['size'][False]>=20)&(grouped['size'][True]>=20)
    t=grouped.loc[keep].copy()
    assert len(t)>0,'No common-support strata at this width.'
    n0=t['size'][False];n1=t['size'][True];p0=t['sum'][False]/n0;p1=t['sum'][True]/n1
    w=(n0+n1)/(n0+n1).sum()
    diff=float((w*(p1-p0)).sum())
    se=np.sqrt((w**2*(p0*(1-p0)/n0+p1*(1-p1)/n1)).sum())
    details=t.index.to_frame(index=False)
    for key,val in [('n_low_ratio',n0),('n_high_ratio',n1),('fpr_low_ratio',p0),('fpr_high_ratio',p1),('weight',w)]:
        details[key]=val.to_numpy()
    details.to_json(OUT/f'H2_strata_width_{width:.2f}.json',orient='records',indent=2)
    return dict(band_width=width,strata_retained=len(t),eligible_negatives=len(neg),
        retained_negatives=int((n0+n1).sum()),coverage=float((n0+n1).sum()/len(neg)),
        coverage_low_ratio=float(n0.sum()/(~neg.high_ratio).sum()),coverage_high_ratio=float(n1.sum()/neg.high_ratio.sum()),
        fpr_low_ratio=float((w*p0).sum()),fpr_high_ratio=float((w*p1).sum()),
        adjusted_fpr_difference=diff,difference_lower=diff-z*se,difference_upper=diff+z*se)
h2=save_table('H2_score_adjusted_FPR',[adjusted_comparison(.2),adjusted_comparison(.1)])

d['age_group']=pd.cut(d.age,[0,30,45,60,np.inf],right=False,labels=['Under 30','30 to under 45','45 to under 60','60 and older'])
assert d.age_group.notna().all()
h3=save_table('H3_age_calibration',[dict(group=str(name),**calibration(g)) for name,g in d.groupby('age_group',observed=True)])
young=d[d.age.lt(30)].residual;older=d[d.age.ge(30)].residual
delta=young.mean()-older.mean();se=np.sqrt(young.var(ddof=1)/len(young)+older.var(ddof=1)/len(older))
h3_contrast=dict(young_gap=float(young.mean()),older_gap=float(older.mean()),difference=float(delta),
    lower=float(delta-z*se),upper=float(delta+z*se))
(OUT/'H3_young_older_contrast.json').write_text(json.dumps(h3_contrast,indent=2))
d['probability_bin']=pd.cut(d.predicted_probability,[0,.05,.10,.15,.20,.30,.50,1],include_lowest=True,right=False)
# Include the mathematically possible exact endpoint p=1 in the last bin.
if d.predicted_probability.eq(1).any():d.loc[d.predicted_probability.eq(1),'probability_bin']=d.probability_bin.cat.categories[-1]
reliability=save_table('H3_age_reliability',[
    dict(age_group=str(age),probability_bin=str(b),**calibration(g))
    for (age,b),g in d.groupby(['age_group','probability_bin'],observed=True)])
overall=calibration(d)
(OUT/'overall_calibration.json').write_text(json.dumps(overall,indent=2,default=float))

# Plot existing results; no fitted calibration transformation is applied.
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,5),layout='constrained')
for age,g in reliability.groupby('age_group',sort=False):
    g=g[g.n>=100]
    ax.plot(g.mean_prediction,g.observed_rate,'o-',label=age)
ax.plot([0,1],[0,1],'--',color='gray',label='Perfect calibration')
ax.set(xlim=(0,.6),ylim=(0,.6),xlabel='Mean predicted repayment-difficulty probability',ylabel='Observed repayment-difficulty rate',title='Blackbox calibration by age (bins with at least 100 applicants)')
ax.legend();fig.savefig(OUT/'age_calibration.png',dpi=160);display(fig);plt.close(fig)

a=h1.iloc[0];b=h2.iloc[0];s=h2.iloc[1];y=h3.iloc[0]



H1_external_score_calibration

H2_ratio_context

H2_score_adjusted_FPR

H3_age_calibration

H3_age_reliability


Matplotlib is building the font cache; this may take a moment.


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,EXT2 < 0.11,2537,584,0.23672,0.23019,0.00652,-0.00922,0.02227,0.59498,0.79623,1953,1162,False
1,0.11 <= EXT2 < 0.25,4865,762,0.15695,0.15663,0.00032,-0.00945,0.01010,0.32220,0.65354,4103,1322,False
2,0.25 <= EXT2 < 0.50,15712,1506,0.10555,0.09585,0.00970,0.00523,0.01417,0.15360,0.40837,14206,2182,False
3,EXT2 >= 0.50,37750,2062,0.05517,0.05462,0.00055,-0.00169,0.00279,0.03774,0.18041,35688,1347,False
4,EXT2 missing,138,11,0.07068,0.07971,-0.00903,-0.05450,0.03645,0.06299,0.09091,127,8,False


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Credit/income <= 4,37586,3089,0.08575,0.08218,0.00357,0.00091,0.00623,0.11769,0.41114,34497,4060,False
1,Credit/income > 4,23416,1836,0.08080,0.07841,0.00239,-0.00091,0.00570,0.09087,0.37092,21580,1961,False


,band_width,strata_retained,eligible_negatives,retained_negatives,coverage,coverage_low_ratio,coverage_high_ratio,fpr_low_ratio,fpr_high_ratio,adjusted_fpr_difference,difference_lower,difference_upper
0,0.2,105,56077,54720,0.97580,0.97287,0.98049,0.10318,0.09893,-0.00425,-0.00850,-0.00000
1,0.1,222,56077,45769,0.81618,0.80262,0.83786,0.08549,0.07769,-0.00779,-0.01193,-0.00366


,group,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Under 30,8900,1007,0.12365,0.11315,0.01051,0.00424,0.01677,0.22020,0.55114,7893,1738,False
1,30 to under 45,24675,2246,0.09016,0.09102,-0.00086,-0.00429,0.00257,0.12350,0.42297,22429,2770,False
2,45 to under 60,20531,1340,0.07071,0.06527,0.00544,0.00216,0.00872,0.07055,0.30373,19191,1354,False
3,60 and older,6896,332,0.04902,0.04814,0.00088,-0.00409,0.00585,0.02422,0.11747,6564,159,False


,age_group,probability_bin,n,events,mean_prediction,observed_rate,calibration_gap,gap_lower,gap_upper,false_positive_rate,recall,negatives,false_positives,small_group
0,Under 30,"[0.0, 0.05)",2236,55,0.02970,0.02460,0.00510,-0.00131,0.01151,0.00000,0.00000,2181,0,False
1,Under 30,"[0.05, 0.1)",2396,155,0.07299,0.06469,0.00830,-0.00153,0.01814,0.00000,0.00000,2241,0,False
2,Under 30,"[0.1, 0.15)",1587,181,0.12344,0.11405,0.00938,-0.00626,0.02503,0.00000,0.00000,1406,0,False
3,Under 30,"[0.15, 0.2)",1050,177,0.17368,0.16857,0.00511,-0.01754,0.02776,0.62543,0.65537,873,546,False
4,Under 30,"[0.2, 0.3)",1026,221,0.24179,0.21540,0.02639,0.00138,0.05140,1.00000,1.00000,805,805,False
5,Under 30,"[0.3, 0.5)",545,174,0.36577,0.31927,0.04650,0.00744,0.08556,1.00000,1.00000,371,371,False
6,Under 30,"[0.5, 1.0)",60,44,0.55875,0.73333,-0.17458,-0.28877,-0.06039,1.00000,1.00000,16,16,True
7,30 to under 45,"[0.0, 0.05)",10644,311,0.02601,0.02922,-0.00321,-0.00640,-0.00002,0.00000,0.00000,10333,0,False
8,30 to under 45,"[0.05, 0.1)",6279,458,0.07221,0.07294,-0.00073,-0.00715,0.00569,0.00000,0.00000,5821,0,False
9,30 to under 45,"[0.1, 0.15)",3308,412,0.12197,0.12455,-0.00258,-0.01384,0.00868,0.00000,0.00000,2896,0,False


<Figure size 700x500 with 1 Axes>

In [8]:
def wilson(k,n):
    if not n:return (np.nan,np.nan)
    r=k/n;den=1+z*z/n;center=(r+z*z/(2*n))/den
    half=z*np.sqrt(r*(1-r)/n+z*z/(4*n*n))/den
    return center-half,center+half

rows=[]
def compare(hypothesis,label,data,eligible,flag,metric):
    base=data.loc[eligible].copy();mask=flag.loc[base.index]
    outcome=(base.flagged.ne(base.TARGET)) if metric=='error_rate' else (base.flagged if metric=='FPR' else ~base.flagged)
    a=outcome[mask];b=outcome[~mask];na=len(a);nb=len(b);ka=int(a.sum());kb=int(b.sum())
    pa=ka/na if na else np.nan;pb=kb/nb if nb else np.nan
    la,ua=wilson(ka,na);lb,ub=wilson(kb,nb)
    delta=pa-pb
    rows.append(dict(hypothesis=hypothesis,comparison=label,metric=metric,n_group=na,errors_group=ka,rate_group=pa,
        n_reference=nb,errors_reference=kb,rate_reference=pb,difference=delta,
        lower=delta-np.sqrt((pa-la)**2+(ub-pb)**2),upper=delta+np.sqrt((ua-pa)**2+(pb-lb)**2)))

def evaluate(frame,population):
    start=len(rows);pos=frame.TARGET.eq(1);neg=frame.TARGET.eq(0);allrows=pd.Series(True,index=frame.index)
    older=frame.age.ge(45);high=frame.EXT_SOURCE_2.ge(.5);young=frame.age.lt(30);low=frame.EXT_SOURCE_3.lt(.25)
    compare('H4','Older high EXT2 versus other positives',frame,pos,older&high,'FNR')
    compare('H4','Older versus younger within high EXT2 positives',frame,pos&high,older,'FNR')
    compare('H4','High versus lower EXT2 within older positives',frame,pos&older&frame.EXT_SOURCE_2.notna(),high,'FNR')
    compare('H5','Younger low EXT3 versus other negatives',frame,neg,young&low,'FPR')
    compare('H5','Younger versus older within low EXT3 negatives',frame,neg&low,young,'FPR')
    compare('H5','Low versus higher EXT3 within young negatives',frame,neg&young&frame.EXT_SOURCE_3.notna(),low,'FPR')
    flags={
        'H6':frame[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].isna().any(axis=1),
        'H7':(frame.predicted_probability-info['threshold']).abs().le(.02),
        'H8':(frame.EXT_SOURCE_2.ge(.5)&frame.EXT_SOURCE_3.lt(.25))|(frame.EXT_SOURCE_3.ge(.5)&frame.EXT_SOURCE_2.lt(.25)),
        'H9':frame.DAYS_EMPLOYED.eq(365243)}
    names={'H6':'Any missing external score versus complete','H7':'Near cutoff versus farther away','H8':'Mixed EXT2 EXT3 versus other observed pairs','H9':'Employment sentinel versus other values'}
    for h,flag in flags.items():
        elig=frame.EXT_SOURCE_2.notna()&frame.EXT_SOURCE_3.notna() if h=='H8' else allrows
        for metric,mask in [('error_rate',elig),('FPR',elig&neg),('FNR',elig&pos)]:
            compare(h,names[h],frame,mask,flag,metric)
    for r in rows[start:]:r['population']=population

# Join the extra audit attribute without changing membership or predictions.
employment=pd.read_csv(ROOT/'application_train.csv',usecols=['SK_ID_CURR','DAYS_EMPLOYED'])
d=d.merge(employment,on='SK_ID_CURR',validate='one_to_one')
evaluate(d,'blackbox')
review_dir=ROOT/'.project2_runtime/error_analysis'
ey=pd.read_csv(review_dir/'eyeball_predictions.csv').copy()
assert len(ey)==500 and set(ey.SK_ID_CURR)==eye_ids
ey['age']=-ey.DAYS_BIRTH/365.25;ey['flagged']=ey.predicted_probability.ge(info['threshold'])
ey['residual']=ey.predicted_probability-ey.TARGET
evaluate(ey,'eyeball')
additional=pd.DataFrame(rows)
additional.to_json(OUT/'additional_hypothesis_results.json',orient='records',indent=2)
display(additional[additional.population.eq('blackbox')].round(4))
motivation=[]
for name,flag in [('H1',ey.EXT_SOURCE_2.lt(.11)),('H2',ey.AMT_CREDIT/ey.AMT_INCOME_TOTAL>4),('H3',ey.age.lt(30))]:
    g=ey[flag]
    motivation.append(dict(hypothesis=name,**calibration(g)))
pd.DataFrame(motivation).to_json(OUT/'original_hypothesis_eyeball_context.json',orient='records',indent=2)
shares={}
for name,f in [('eyeball',ey),('blackbox',d)]:
    err=f.flagged.ne(f.TARGET);near=(f.predicted_probability-info['threshold']).abs().le(.02)
    shares[name]=dict(errors=int(err.sum()),near_errors=int((err&near).sum()),share=float((err&near).sum()/err.sum()))
(OUT/'near_threshold_error_shares.json').write_text(json.dumps(shares,indent=2))
assert len(d)==61002 and set(d.SK_ID_CURR).isdisjoint(eye_ids)
assert len(ey)==500 and ey.flagged.ne(ey.TARGET).sum()==len(worksheet)
print('Near-threshold shares:',shares)


Near-threshold shares: {'eyeball': {'errors': 81, 'near_errors': 8, 'share': 0.09876543209876543}, 'blackbox': {'errors': 8995, 'near_errors': 1518, 'share': 0.16876042245692052}}


,hypothesis,comparison,metric,n_group,errors_group,rate_group,n_reference,errors_reference,rate_reference,difference,lower,upper,population
0,H4,Older high EXT2 versus other positives,FNR,785,704,0.8968,4140,2270,0.5483,0.3485,0.3208,0.3731,blackbox
1,H4,Older versus younger within high EXT2 positives,FNR,785,704,0.8968,1277,986,0.7721,0.1247,0.0926,0.1554,blackbox
2,H4,High versus lower EXT2 within older positives,FNR,785,704,0.8968,884,519,0.5871,0.3097,0.2701,0.3478,blackbox
3,H5,Younger low EXT3 versus other negatives,FPR,1037,554,0.5342,55040,5467,0.0993,0.4349,0.4044,0.4652,blackbox
4,H5,Younger versus older within low EXT3 negatives,FPR,1037,554,0.5342,3736,1512,0.4047,0.1295,0.0952,0.1635,blackbox
5,H5,Low versus higher EXT3 within young negatives,FPR,1037,554,0.5342,4774,683,0.1431,0.3912,0.3591,0.4228,blackbox
6,H6,Any missing external score versus complete,error_rate,39269,6102,0.1554,21733,2893,0.1331,0.0223,0.0165,0.0280,blackbox
7,H6,Any missing external score versus complete,FPR,35963,4154,0.1155,20114,1867,0.0928,0.0227,0.0175,0.0278,blackbox
8,H6,Any missing external score versus complete,FNR,3306,1948,0.5892,1619,1026,0.6337,-0.0445,-0.0731,-0.0155,blackbox
9,H7,Near cutoff versus farther away,error_rate,3443,1518,0.4409,57559,7477,0.1299,0.3110,0.2943,0.3279,blackbox


## 7 Interpretation and submission

Use the tables to distinguish persistent associations from explanations that were not supported. The final document summarizes the updated results. Do not label missing information, unusual values or every false alarm as a proven cause. Raw employment sentinel values were already cleaned before training.

Submit Final_Error_Analysis.csv after the team checks its factual drafts and adds its comments. Supporting all500 predictions and correct-case comparisons are in .project2_runtime/error_analysis; they are not extra errors. Changes to the model or threshold require a new review version. This notebook and the main notebook are reproducible technical deliverables; manual signoff is not performed automatically.
